# Validación cruzada estratificada — Clasificador MTSK

Este notebook reproduce el protocolo del clasificador afinado (Experimento 13) bajo **validación cruzada estratificada de 5 pliegues** y evalúa, **con los mismos pliegues**, tres encoders multilingües y una línea base clásica.

**Protocolo por pliegue**
1. El pliegue externo de prueba (20 %) nunca se usa para entrenar ni para seleccionar la época.
2. Dentro del 80 % restante se reserva una **validación interna** (≈15 %) para la parada temprana y la selección de la mejor época.
3. El modelo con la mejor época se evalúa una sola vez sobre el pliegue de prueba.

**Configuración idéntica al Experimento 13:** prefijo `query:` (solo e5), concatenación título + resumen + palabras clave, 128 tokens, agrupamiento por el primer token ([CLS]), Dropout(0.3) + capa lineal, entropía cruzada con pesos de clase *balanced*, AdamW (lr = 5e-5), calentamiento lineal del 10 %, lote 16, máximo 20 épocas, paciencia 3.

**Pliegues:** `StratifiedGroupKFold` **sin barajado** (`shuffle=False`). En scikit-learn 1.6 la opción `shuffle=True` produce pliegues mal estratificados; sin barajado la partición es determinista, está bien estratificada y es idéntica en cualquier versión.

**Salidas** (carpeta `outputs_cv/`): predicciones fuera de pliegue con probabilidades, métricas por pliegue, resumen con IC (t y *bootstrap*), métricas por categoría, por idioma y matrices de confusión.

## 1 · Instalación e importaciones

In [ ]:
!pip install -q transformers scikit-learn openpyxl langdetect matplotlib seaborn scipy

In [ ]:
import os, gc, json, random, time
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             classification_report, confusion_matrix)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
import transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

## 2 · Configuración

In [ ]:
# ── Datos ──────────────────────────────────────────────────────────────────
DATA_PATH  = "data/corpus_MTSK_depurado.xlsx"   # ← corpus ya depurado de duplicados
TEXT_COLS  = ("Título", "Resumen (texto literal)", "Palabras clave (texto literal)")
LABEL_COL  = "Clasificacion tematica"
GROUP_COL  = "Grupo"    # OPCIONAL: mismo valor para versiones relacionadas de un trabajo
                        # (p. ej., ponencia y artículo) que decidan conservar; así nunca
                        # quedan separadas entre entrenamiento y prueba. Si no existe, se ignora.
LANG_COL   = "Idioma"   # OPCIONAL: si no existe, se detecta automáticamente sobre el resumen

# ── Protocolo ──────────────────────────────────────────────────────────────
N_FOLDS    = 5
SEED       = 7
INNER_VAL  = 0.15       # fracción aproximada del pliegue de entrenamiento para validación interna
N_BOOT     = 2000       # remuestreos para el IC bootstrap sobre predicciones agrupadas

# ── Hiperparámetros (idénticos al Experimento 13) ──────────────────────────
MAX_LEN    = 128
BATCH_SIZE = 16
LR         = 5e-5
MAX_EPOCHS = 20
PATIENCE   = 3
DROPOUT    = 0.3
WARMUP     = 0.10

# ── Modelos: nombre corto → (id en Hugging Face, prefijo de entrada) ───────
# Cada modelo: id en Hugging Face, prefijo, longitud máxima, tasa de aprendizaje,
# tamaño de lote por paso y pasos de acumulación (lote efectivo = lote × acumulación = 16)
MODELS = {
    "e5-large":      dict(hf="intfloat/multilingual-e5-large", prefix="query: ", max_len=128, lr=LR, batch=16, accum=1),
    "xlmr-base":     dict(hf="xlm-roberta-base", prefix="", max_len=128, lr=LR, batch=16, accum=1),
    "xlmr-base-2e5": dict(hf="xlm-roberta-base", prefix="", max_len=128, lr=2e-5, batch=16, accum=1),
    "minilm":        dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", prefix="", max_len=128, lr=LR, batch=16, accum=1),
    "e5-large-256":  dict(hf="intfloat/multilingual-e5-large", prefix="query: ", max_len=256, lr=LR, batch=8, accum=2),
    "e5-large-512":  dict(hf="intfloat/multilingual-e5-large", prefix="query: ", max_len=512, lr=LR, batch=4, accum=4),
}
# Principal: e5-large, xlmr-base, minilm (mismos hiperparámetros para todos).
# Complementarios: xlmr-base-2e5 (sensibilidad a la tasa de aprendizaje) y e5-large-256 / -512 (efecto del truncamiento).
RUN_MODELS = ["e5-large", "xlmr-base", "minilm", "xlmr-base-2e5", "e5-large-256"]   # añade "e5-large-512" si hay tiempo y GPU
RUN_TFIDF  = True

OUT_DIR = "outputs_cv"
os.makedirs(OUT_DIR, exist_ok=True)

## 3 · Datos, idioma y grupos

In [ ]:
df = pd.read_excel(DATA_PATH).reset_index(drop=True)
print("Documentos:", len(df))

def build_texts(frame, prefix=""):
    t = (frame[TEXT_COLS[0]].fillna("").astype(str) + " " +
         frame[TEXT_COLS[1]].fillna("").astype(str) + " " +
         frame[TEXT_COLS[2]].fillna("").astype(str))
    return (prefix + t).str.strip().tolist()

label_names = sorted(df[LABEL_COL].unique().tolist())
label2idx   = {l: i for i, l in enumerate(label_names)}
y           = np.array([label2idx[l] for l in df[LABEL_COL]])
print("Distribución:", df[LABEL_COL].value_counts().sort_index().to_dict())

# Idioma
if LANG_COL in df.columns and df[LANG_COL].notna().all():
    lang = df[LANG_COL].astype(str).values
else:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    def _detect(s):
        try:
            return detect(s) if s.strip() else "desconocido"
        except Exception:
            return "desconocido"
    src  = df[TEXT_COLS[1]].fillna("").astype(str)
    src  = src.where(src.str.strip() != "", df[TEXT_COLS[0]].fillna("").astype(str))
    lang = src.apply(_detect).values
    df[LANG_COL] = lang
print("Idiomas:", pd.Series(lang).value_counts().to_dict())
print(pd.crosstab(df[LABEL_COL], df[LANG_COL], margins=True))

# Grupos (cada documento es su propio grupo salvo que se indique lo contrario)
if GROUP_COL in df.columns:
    g = df[GROUP_COL].astype(object)
    groups = np.array([f"g_{v}" if pd.notna(v) and str(v).strip() else f"doc_{i}"
                       for i, v in enumerate(g)])
else:
    groups = np.array([f"doc_{i}" for i in range(len(df))])
print("Grupos:", len(set(groups)))

## 4 · Pliegues (compartidos por todos los modelos)

In [ ]:
outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=False)
folds = []
for k, (trval_idx, test_idx) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
    n_inner = max(2, int(round(1 / INNER_VAL)))
    inner   = StratifiedGroupKFold(n_splits=n_inner, shuffle=False)
    tr_rel, va_rel = next(inner.split(np.zeros(len(trval_idx)), y[trval_idx], groups[trval_idx]))
    folds.append({"fold": k,
                  "train": trval_idx[tr_rel].tolist(),
                  "val":   trval_idx[va_rel].tolist(),
                  "test":  test_idx.tolist()})
    print(f"Pliegue {k}: train={len(tr_rel)} val={len(va_rel)} test={len(test_idx)} | "
          f"test por clase={np.bincount(y[test_idx], minlength=len(label_names)).tolist()} | "
          f"val por clase={np.bincount(y[trval_idx[va_rel]], minlength=len(label_names)).tolist()}")

# Comprobaciones: cada documento aparece exactamente una vez como prueba y sin solapamientos
all_test = np.concatenate([f["test"] for f in folds])
assert len(all_test) == len(y) and len(set(all_test)) == len(y)
for f in folds:
    assert not (set(f["train"]) & set(f["test"])) and not (set(f["val"]) & set(f["test"]))
    assert not (set(groups[f["train"]]) & set(groups[f["test"]])), "Un grupo quedó repartido"

fold_of = np.empty(len(y), dtype=int)
for f in folds:
    fold_of[f["test"]] = f["fold"]
json.dump(folds, open(f"{OUT_DIR}/folds.json", "w"))
pd.DataFrame([{"pliegue": f["fold"], "conjunto": part, **{label_names[c]: int((y[f[part]] == c).sum()) for c in range(len(label_names))}}
              for f in folds for part in ["train", "val", "test"]]).to_csv(f"{OUT_DIR}/composicion_pliegues.csv", index=False)
print("Pliegues guardados ✅")

## 5 · Longitud de las entradas y truncamiento

In [ ]:
trunc_rows = []
for short in RUN_MODELS:
    cfg = MODELS[short]; hf_id, prefix, max_len = cfg["hf"], cfg["prefix"], cfg["max_len"]
    tok  = AutoTokenizer.from_pretrained(hf_id)
    lens = np.array([len(tok(t)["input_ids"]) for t in build_texts(df, prefix)])
    kw_start = np.array([len(tok((prefix + str(a) + " " + str(b)).strip())["input_ids"])
                         for a, b in zip(df[TEXT_COLS[0]].fillna(""), df[TEXT_COLS[1]].fillna(""))])
    has_kw = df[TEXT_COLS[2]].fillna("").astype(str).str.strip() != ""
    trunc_rows.append({
        "modelo": short, "max_len": max_len,
        "tokens_media": round(lens.mean(), 1),
        "tokens_mediana": float(np.median(lens)),
        "pct_truncados": round(100 * (lens > max_len).mean(), 1),
        "docs_con_palabras_clave": int(has_kw.sum()),
        "palabras_clave_visibles": int(((kw_start < max_len) & has_kw).sum()),
    })
trunc_df = pd.DataFrame(trunc_rows)
trunc_df.to_csv(f"{OUT_DIR}/truncamiento.csv", index=False)
trunc_df

## 6 · Modelo y entrenamiento por pliegue

In [ ]:
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class TextDS(Dataset):
    def __init__(self, enc, labels):
        self.enc, self.labels = enc, labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(int(self.labels[i]))
        return item

class Classifier(torch.nn.Module):
    # Igual que MTSKClassifier del Experimento 13: [CLS] → Dropout → Linear
    def __init__(self, backbone, num_labels, p=DROPOUT):
        super().__init__()
        self.backbone   = backbone
        self.dropout    = torch.nn.Dropout(p)
        self.classifier = torch.nn.Linear(backbone.config.hidden_size, num_labels)
    def forward(self, input_ids, attention_mask):
        h = self.backbone(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(h))

@torch.no_grad()
def predict_proba(model, loader):
    model.eval(); out = []
    for b in loader:
        logits = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        out.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.concatenate(out)

def run_fold(cfg, tok, texts, fold, seed):
    set_seed(seed)
    hf_id, max_len, lr, bs, accum = cfg["hf"], cfg["max_len"], cfg["lr"], cfg["batch"], cfg["accum"]
    tr, va, te = fold["train"], fold["val"], fold["test"]
    enc = lambda idx: tok([texts[i] for i in idx], truncation=True, padding=True, max_length=max_len)
    g = torch.Generator(); g.manual_seed(seed)
    tr_loader = DataLoader(TextDS(enc(tr), y[tr]), batch_size=bs, shuffle=True, generator=g)
    va_loader = DataLoader(TextDS(enc(va), y[va]), batch_size=bs, shuffle=False)
    te_loader = DataLoader(TextDS(enc(te), y[te]), batch_size=bs, shuffle=False)

    w = compute_class_weight("balanced", classes=np.arange(len(label_names)), y=y[tr])
    w = torch.tensor(w, dtype=torch.float).to(DEVICE)

    model = Classifier(AutoModel.from_pretrained(hf_id), len(label_names)).to(DEVICE)
    opt   = AdamW(model.parameters(), lr=lr)
    steps_per_epoch = int(np.ceil(len(tr_loader) / accum))
    total = steps_per_epoch * MAX_EPOCHS
    sch   = get_linear_schedule_with_warmup(opt, int(total * WARMUP), total)

    best_f1, best_epoch, bad, hist = -1.0, 0, 0, []
    ckpt = f"/tmp/best_fold{fold['fold']}.pt"
    for ep in range(1, MAX_EPOCHS + 1):
        model.train(); tot = 0.0; opt.zero_grad()
        for step, b in enumerate(tr_loader, 1):
            loss = torch.nn.functional.cross_entropy(
                model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE)),
                b["labels"].to(DEVICE), weight=w)
            (loss / accum).backward(); tot += loss.item()
            if step % accum == 0 or step == len(tr_loader):
                opt.step(); sch.step(); opt.zero_grad()
        pv  = predict_proba(model, va_loader).argmax(1)
        vf1 = f1_score(y[va], pv, average="macro", zero_division=0)
        hist.append({"epoch": ep, "train_loss": tot / len(tr_loader), "val_macro_f1": vf1})
        if vf1 > best_f1:
            best_f1, best_epoch, bad = vf1, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    probs = predict_proba(model, te_loader)
    del model, opt; gc.collect(); torch.cuda.empty_cache(); os.remove(ckpt)
    return probs, best_epoch, best_f1, hist

## 7 · Métricas

In [ ]:
def metrics(y_true, y_pred):
    return {"accuracy":  accuracy_score(y_true, y_pred),
            "macro_f1":  f1_score(y_true, y_pred, average="macro", zero_division=0),
            "precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "recall":    recall_score(y_true, y_pred, average="macro", zero_division=0)}

def t_ci(vals, conf=0.95):
    vals = np.asarray(vals); m, s = vals.mean(), vals.std(ddof=1)
    h = stats.t.ppf((1 + conf) / 2, len(vals) - 1) * s / np.sqrt(len(vals))
    return m, s, m - h, m + h

def boot_ci(y_true, y_pred, fn, n=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n_obs = len(y_true); vals = []
    for _ in range(n):
        i = rng.integers(0, n_obs, n_obs)
        vals.append(fn(y_true[i], y_pred[i]))
    return np.percentile(vals, [2.5, 97.5])

macro_f1 = lambda a, b: f1_score(a, b, average="macro", zero_division=0)

## 8 · Ejecución

In [ ]:
results = {}   # modelo → {"probs": (n, C), "fold_metrics": [...], "epochs": [...]}

for short in RUN_MODELS:
    cfg = MODELS[short]
    print(f"\n══════ {short} ({cfg['hf']}, max_len={cfg['max_len']}, lr={cfg['lr']}) ══════")
    tok   = AutoTokenizer.from_pretrained(cfg["hf"])
    texts = build_texts(df, cfg["prefix"])
    probs = np.zeros((len(y), len(label_names)))
    fold_metrics, epochs, histories = [], [], []
    for f in folds:
        t0 = time.time()
        p, be, vf1, hist = run_fold(cfg, tok, texts, f, SEED + f["fold"])
        probs[f["test"]] = p
        m = metrics(y[f["test"]], p.argmax(1)); m["fold"] = f["fold"]
        fold_metrics.append(m); epochs.append(be)
        histories.append({"fold": f["fold"], "history": hist})
        print(f"  Pliegue {f['fold']}: mejor época={be} (val F1={vf1:.4f}) | "
              f"test macro-F1={m['macro_f1']:.4f} acc={m['accuracy']:.4f} | {time.time()-t0:.0f}s")
    results[short] = {"probs": probs, "fold_metrics": fold_metrics, "epochs": epochs}
    json.dump(histories, open(f"{OUT_DIR}/{short}_curvas.json", "w"))

if RUN_TFIDF:
    print("\n══════ TF-IDF + regresión logística ══════")
    texts = build_texts(df, "")
    probs = np.zeros((len(y), len(label_names))); fold_metrics = []
    for f in folds:
        trv = f["train"] + f["val"]   # sin parada temprana: usa todo el 80 %
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
        Xtr = vec.fit_transform([texts[i] for i in trv])
        Xte = vec.transform([texts[i] for i in f["test"]])
        clf = LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0, random_state=SEED)
        clf.fit(Xtr, y[trv])
        probs[f["test"]] = clf.predict_proba(Xte)
        m = metrics(y[f["test"]], probs[f["test"]].argmax(1)); m["fold"] = f["fold"]
        fold_metrics.append(m)
        print(f"  Pliegue {f['fold']}: macro-F1={m['macro_f1']:.4f} acc={m['accuracy']:.4f}")
    results["tfidf-lr"] = {"probs": probs, "fold_metrics": fold_metrics, "epochs": [None] * N_FOLDS}

## 9 · Resultados y exportación

In [ ]:
summary_rows, per_fold_rows = [], []
for name, r in results.items():
    pred = r["probs"].argmax(1)
    for m, e in zip(r["fold_metrics"], r["epochs"]):
        per_fold_rows.append({"modelo": name, **m, "mejor_epoca": e})
    row = {"modelo": name}
    for key in ["accuracy", "macro_f1", "precision", "recall"]:
        mean, sd, lo, hi = t_ci([m[key] for m in r["fold_metrics"]])
        row.update({f"{key}_media": mean, f"{key}_de": sd, f"{key}_ic_t_inf": lo, f"{key}_ic_t_sup": hi})
    lo, hi = boot_ci(y, pred, macro_f1)
    row.update({"macro_f1_agrupado": macro_f1(y, pred), "macro_f1_ic_boot_inf": lo, "macro_f1_ic_boot_sup": hi})
    summary_rows.append(row)

    # Predicciones fuera de pliegue
    oof = pd.DataFrame(r["probs"], columns=[f"p_{l}" for l in label_names])
    oof.insert(0, "indice", np.arange(len(y)))
    oof.insert(1, "pliegue", fold_of)
    oof.insert(2, "idioma", df[LANG_COL].values)
    oof.insert(3, "etiqueta", [label_names[i] for i in y])
    oof.insert(4, "prediccion", [label_names[i] for i in pred])
    oof.insert(5, "confianza", r["probs"].max(1))
    oof.to_csv(f"{OUT_DIR}/{name}_predicciones_oof.csv", index=False)

    # Métricas por categoría (sobre las predicciones agrupadas)
    rep = classification_report(y, pred, target_names=label_names, digits=4, output_dict=True, zero_division=0)
    pd.DataFrame(rep).T.to_csv(f"{OUT_DIR}/{name}_por_categoria.csv")

    # Métricas por idioma
    lang_rows = []
    for lg in sorted(set(df[LANG_COL])):
        msk = df[LANG_COL].values == lg
        lang_rows.append({"idioma": lg, "n": int(msk.sum()),
                          "accuracy": accuracy_score(y[msk], pred[msk]),
                          "macro_f1": macro_f1(y[msk], pred[msk])})
    pd.DataFrame(lang_rows).to_csv(f"{OUT_DIR}/{name}_por_idioma.csv", index=False)

    # Matriz de confusión normalizada por fila
    cm = confusion_matrix(y, pred, labels=range(len(label_names)))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cmn, annot=cm, fmt="d", cmap="Blues", vmin=0, vmax=1,
                xticklabels=label_names, yticklabels=label_names)
    plt.xlabel("Predicción"); plt.ylabel("Etiqueta"); plt.title(f"{name} — validación cruzada (n={len(y)})")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/{name}_matriz_confusion.png", dpi=200); plt.show()

summary = pd.DataFrame(summary_rows)
summary.to_csv(f"{OUT_DIR}/resumen.csv", index=False)
pd.DataFrame(per_fold_rows).to_csv(f"{OUT_DIR}/metricas_por_pliegue.csv", index=False)
cols = ["modelo", "macro_f1_media", "macro_f1_de", "macro_f1_ic_t_inf", "macro_f1_ic_t_sup",
        "macro_f1_agrupado", "macro_f1_ic_boot_inf", "macro_f1_ic_boot_sup", "accuracy_media"]
summary[cols].round(4)

In [ ]:
# Comparación pareada: diferencia de macro-F1 entre e5-large y cada modelo, sobre los MISMOS documentos
# (bootstrap pareado de las predicciones fuera de pliegue). Si el IC no incluye 0, la diferencia es robusta.
REF = "e5-large"
if REF in results:
    rng = np.random.default_rng(SEED); n_obs = len(y); rows = []
    pred_ref = results[REF]["probs"].argmax(1)
    for name, r in results.items():
        if name == REF:
            continue
        pred_o = r["probs"].argmax(1); diffs = []
        for _ in range(N_BOOT):
            i = rng.integers(0, n_obs, n_obs)
            diffs.append(macro_f1(y[i], pred_ref[i]) - macro_f1(y[i], pred_o[i]))
        rows.append({"comparacion": f"{REF} - {name}", "diferencia": macro_f1(y, pred_ref) - macro_f1(y, pred_o),
                     "ic_inf": np.percentile(diffs, 2.5), "ic_sup": np.percentile(diffs, 97.5),
                     "prop_boot_favorable": float(np.mean(np.array(diffs) > 0))})
    comp = pd.DataFrame(rows); comp.to_csv(f"{OUT_DIR}/comparacion_pareada.csv", index=False)
    display(comp.round(4))

In [ ]:
# Configuración usada, para el apéndice o el repositorio
json.dump({"n_docs": int(len(y)), "labels": label_names, "n_folds": N_FOLDS, "seed": SEED,
           "inner_val": INNER_VAL, "max_len": MAX_LEN, "batch_size": BATCH_SIZE, "lr": LR,
           "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "dropout": DROPOUT, "warmup": WARMUP,
           "models": {k: MODELS[k] for k in RUN_MODELS}, "stratified_group_kfold_shuffle": False, "tfidf": RUN_TFIDF,
           "torch": torch.__version__, "transformers": transformers.__version__,
           "sklearn": sklearn.__version__},
          open(f"{OUT_DIR}/configuracion.json", "w"), indent=2, ensure_ascii=False)
print("Todo guardado en", OUT_DIR, "✅")